<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-DataWrangling/blob/main/W03_pdf_extraction_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 PDF-Informationsextraktion — Drei Praxisbeispiele
**Data Wrangling — Praktisches Notebook**

Nicht alle PDFs sind gleich aufgebaut. Dieses Notebook vergleicht, wie vier Python-Bibliotheken mit drei grundlegend verschiedenen PDF-Typen aus der Praxis umgehen:

| # | Datei | Inhalt | Hauptherausforderung |
|---|------|---------|----------------------|
| 1 | `arxiv_2205_09911v2.pdf` | Wissenschaftlicher Artikel (EN) — *Can Foundation Models Wrangle Your Data?* | Zweispaltiges Layout, Tabellen, 106 Referenzen |
| 2 | `swisscom_internetbox_glasfaser_10233695_de.pdf` | Router-Installationsanleitung (DE) | Bildlastig, Schritte teils in Bildern eingebettet |
| 3 | `WEB_RZ_BAI_Cur25_Modulübersichten_250226.pdf` | BSc BAI Modulübersicht (DE) | Farbiges Rasterlayout, räumliche Bedeutung geht bei der Extraktion verloren |

**Verglichene Bibliotheken:**

| Bibliothek | Ansatz |
|------------|--------|
| `pypdf` | Reines Python, leichtgewichtig — gut für Metadaten und schnellen Textzugriff |
| `pdfplumber` | Koordinatenbewusst, beste integrierte Tabellenerkennung |
| `PyMuPDF` (`fitz`) | C-basiert, sehr schnell, verarbeitet Bilder und Rendering |
| `pdfminer.six` | Low-Level-Layoutobjekte — liefert x/y-Positionen jedes Elements |

---
**Verwendung dieses Notebooks:**
- **Teil 1** (arXiv) ist vollständig ausgearbeitet — alle Zellen ausführen und die Kommentare lesen.
- **Teile 2 & 3** sind Übungen — die gleichen Werkzeuge auf zunehmend schwierigere PDFs anwenden.
- **Teil 4** stellt alle Ergebnisse nebeneinander dar.

---
## 🔧 Setup

In [ ]:
!pip install pypdf pdfplumber pymupdf pdfminer.six --quiet
print('Pakete installiert.')

In [ ]:
import re, time, io
from pathlib import Path
from collections import Counter

import pypdf
import pdfplumber
import fitz                                           # PyMuPDF
from pdfminer.high_level import extract_text as pdfminer_extract
from pdfminer.high_level import extract_pages
from pdfminer.layout import LTTextBox, LTFigure, LTRect, LTLine

import pandas as pd
from IPython.display import display, Image as IPImage

print('Imports abgeschlossen.')

## Die drei PDFs hochladen

In [ ]:
from google.colab import files

print('Bitte die drei PDFs hochladen:')
uploaded = files.upload()

pdf_paths = {}
for filename, data in uploaded.items():
    Path(filename).write_bytes(data)
    low = filename.lower()
    if 'arxiv' in low:
        pdf_paths['arxiv'] = filename
    elif 'swisscom' in low:
        pdf_paths['swisscom'] = filename
    else:
        pdf_paths['curriculum'] = filename

for key, path in pdf_paths.items():
    kb = Path(path).stat().st_size / 1024
    print(f'  [{key:12s}] {path}  ({kb:.0f} KB)')

## Hilfsfunktionen
Einmal ausführen — wird im gesamten Notebook verwendet.

In [ ]:
def extraction_report(pdf_path):
    """Alle vier Bibliotheken ausführen; gibt DataFrame zurück: library|pages|chars|words|time_sec."""
    rows = []
    n = len(pypdf.PdfReader(pdf_path).pages)

    t0 = time.perf_counter()
    r = pypdf.PdfReader(pdf_path)
    tx = '\n'.join((r.pages[i].extract_text() or '') for i in range(n))
    rows.append(('pypdf',        n, len(tx),   len(tx.split()),   round(time.perf_counter()-t0,3)))

    t0 = time.perf_counter()
    with pdfplumber.open(pdf_path) as pdf:
        tx = '\n'.join(p.extract_text() or '' for p in pdf.pages)
    rows.append(('pdfplumber',   n, len(tx),   len(tx.split()),   round(time.perf_counter()-t0,3)))

    t0 = time.perf_counter()
    doc = fitz.open(pdf_path)
    tx = '\n'.join(p.get_text() for p in doc); doc.close()
    rows.append(('PyMuPDF',      n, len(tx),   len(tx.split()),   round(time.perf_counter()-t0,3)))

    t0 = time.perf_counter()
    tx = pdfminer_extract(pdf_path)
    rows.append(('pdfminer.six', n, len(tx),   len(tx.split()),   round(time.perf_counter()-t0,3)))

    df = pd.DataFrame(rows, columns=['library','pages','chars','words','time_sec'])
    df['chars_per_sec'] = (df['chars']/df['time_sec']).astype(int)
    return df


def get_text(pdf_path, library='pymupdf'):
    """Volltext mit der gewählten Bibliothek extrahieren."""
    if library == 'pymupdf':
        doc = fitz.open(pdf_path)
        t = '\n'.join(p.get_text() for p in doc); doc.close(); return t
    if library == 'pdfplumber':
        with pdfplumber.open(pdf_path) as pdf:
            return '\n'.join(p.extract_text() or '' for p in pdf.pages)
    if library == 'pypdf':
        r = pypdf.PdfReader(pdf_path)
        return '\n'.join((r.pages[i].extract_text() or '') for i in range(len(r.pages)))
    if library == 'pdfminer':
        return pdfminer_extract(pdf_path)


def count_images(pdf_path):
    """Bilder pro Seite via PyMuPDF."""
    doc = fitz.open(pdf_path)
    rows = [{'page':i+1,'n_images':len(p.get_images(full=True))} for i,p in enumerate(doc)]
    doc.close()
    return pd.DataFrame(rows)


def layout_elements(pdf_path, page_number=0):
    """pdfminer-Layoutobjekte für eine Seite."""
    rows = []
    for pnum, layout in enumerate(extract_pages(pdf_path)):
        if pnum != page_number: continue
        for el in layout:
            x0,y0,x1,y1 = [round(v,1) for v in el.bbox]
            preview = el.get_text()[:80].replace('\n',' ').strip() if isinstance(el,LTTextBox) else ''
            rows.append({'type':type(el).__name__,'x0':x0,'y0':y0,'x1':x1,'y1':y1,
                         'w':round(x1-x0,1),'h':round(y1-y0,1),'preview':preview})
    return pd.DataFrame(rows)


def render_page(pdf_path, page_number=0, dpi=110):
    """Eine PDF-Seite als PNG rendern und inline anzeigen."""
    doc = fitz.open(pdf_path)
    pix = doc[page_number].get_pixmap(matrix=fitz.Matrix(dpi/72, dpi/72))
    doc.close()
    display(IPImage(data=pix.tobytes('png'), width=720))


print('Hilfsfunktionen bereit.')

---
# Teil 1 — arXiv-Artikel (Vollständig ausgearbeitet)
### *"Can Foundation Models Wrangle Your Data?"* — Narayan et al., Stanford, 2022

Dies ist ein **natives PDF** — der gesamte Text liegt als korrekte Vektorobjekte vor. Die Hauptherausforderungen sind:
- **Zweispaltiges Layout**, das die Lesereihenfolge durcheinanderbringen kann
- **Inline-Mathematiknotation** und Referenzen wie `[20]`
- **6 nummerierte Ergebnistabellen** über das Dokument verteilt

## 1.0 Zuerst visuell inspizieren

Vor der Extraktion immer eine Seite rendern — das zeigt, was man *erwarten* kann zu finden.

In [ ]:
render_page(pdf_paths['arxiv'], page_number=0)

## 1.1 Datei-Metadaten

In [ ]:
pdf = pdf_paths['arxiv']
reader = pypdf.PdfReader(pdf)
print(f'Pages     : {len(reader.pages)}')
print(f'File size : {Path(pdf).stat().st_size/1024:.0f} KB')
print()
print('Eingebettete Metadaten (pypdf):')
for k, v in (reader.metadata or {}).items():
    if v: print(f'  {k:20s}: {v}')

## 1.2 Bibliotheksvergleich — Ausbeute und Geschwindigkeit

In [ ]:
rep = extraction_report(pdf_paths['arxiv'])
display(rep)

spread = rep['chars'].max() - rep['chars'].min()
print(f"\nStreuung zwischen Bibliotheken: {spread:,} Zeichen  ({spread/rep['chars'].mean()*100:.1f}% des Mittelwerts)")
print('Kleine Streuung = Bibliotheken stimmen überein = sauberes natives PDF.')

## 1.3 Das Zweispalten-Problem

PDF-Bibliotheken lesen Textobjekte in der Speicherreihenfolge, nicht in der visuellen Reihenfolge. Bei einem zweispaltigen Layout kann Text aus der rechten Spalte mitten in Sätzen der linken Spalte auftauchen.

Die `pdfminer`-Layoutobjekte machen das *sichtbar*: Man kann abfragen, *wo* sich jede Textbox auf der Seite befindet und ob die Extraktionsreihenfolge der Spaltenstruktur folgt.

In [ ]:
# Seite 2 untersuchen — erste vollständig zweispaltige Seite
layout = layout_elements(pdf_paths['arxiv'], page_number=2)
print(layout)
tboxes = layout[layout['type']=='LTTextBoxHorizontal'].copy()

# PDF-Seitenbreite ≈ 612 Punkte; Spaltengrenze ≈ x=310
tboxes['column'] = tboxes['x0'].apply(lambda x: 'LEFT' if x < 310 else 'RIGHT')

print(f'Textboxen auf Seite 2: {len(tboxes)}')
print(f'Links : {(tboxes.column=="LEFT").sum()}   Rechts: {(tboxes.column=="RIGHT").sum()}')
print()
print('Extraktionsreihenfolge — beachte Spaltenwechsel auf gleicher y0-Höhe:')
print(tboxes[['column','x0','y0','preview']].head(14).to_string(index=False))

In [ ]:
# Einfache Korrektur: Linke Spalte von oben nach unten sortieren, dann rechte Spalte
# (y0 steigt nach unten in PDF-Koordinaten, daher aufsteigend sortieren)
left_col  = tboxes[tboxes.column=='LEFT' ].sort_values('y0')
right_col = tboxes[tboxes.column=='RIGHT'].sort_values('y0')

corrected_text = '\n'.join(
    list(left_col['preview']) + list(right_col['preview'])
)

print('Erste 600 Zeichen des spaltenkorrigierten Texts (Seite 2):')
print(corrected_text[:600])

## 1.4 Extraktion des Abstracts

Der Abstract ist auf Seite 1 als `ABSTRACT` gekennzeichnet. Wir extrahieren ihn mit Regex aus dem Volltext. Regex (reguläre Ausdrücke) werden wir im Modul noch behandeln.

In [ ]:
full_text = get_text(pdf_paths['arxiv'], library='pymupdf')

abstract_match = re.search(
    r'ABSTRACT\s*\n(.*?)(?=\n1\s+INTRODUCTION)',
    full_text, re.DOTALL
)

if abstract_match:
    abstract = ' '.join(abstract_match.group(1).split())
    print(f'Extrahierter Abstract ({len(abstract)} Zeichen):\n')
    print(abstract)
else:
    print('Muster hat nicht gepasst. Debug: Erste 1500 Zeichen von full_text:')
    print(full_text[:1500])

## 1.5 Abschnittsüberschriften

Das Paper verwendet GROSSGESCHRIEBENE nummerierte Überschriften: `1 INTRODUCTION`, `2 BACKGROUND` usw., sowie Anhangüberschriften wie `A SMALL FM FINETUNING EXPERIMENTS`.

In [ ]:
section_pat = re.compile(
    r'^([\dA-Z]+(?:\.\d+)?)\s{1,4}([A-Z][A-Z0-9,/ &\-]{3,60})$',
    re.MULTILINE
)
sections = section_pat.findall(full_text)
sec_df = pd.DataFrame(sections, columns=['number','heading'])
print(f'Abschnitte gefunden: {len(sec_df)}')
print()
print(sec_df.to_string(index=False))

## 1.6 Tabellenextraktion mit `pdfplumber`

Das Paper enthält **6 nummerierte Tabellen** (Tabellen 1–6) mit F1-Scores, die GPT-3 mit Stand-der-Technik-Baselines vergleichen. `pdfplumber` erkennt Tabellen anhand der Liniengeometrie.

In [ ]:
found_tables = []
with pdfplumber.open(pdf_paths['arxiv']) as pdf:
    for i, page in enumerate(pdf.pages):
        for j, raw in enumerate(page.extract_tables()):
            if raw and len(raw) >= 2 and len(raw[0]) >= 2:
                df = pd.DataFrame(raw[1:], columns=raw[0])
                df.attrs = {'page': i+1}
                found_tables.append(df)

print(f'Tabellen erkannt: {len(found_tables)}  (Paper enthält 6 Tabellen)')
print()
for df in found_tables:
    print(f'--- Page {df.attrs["page"]}  shape {df.shape} ---')
    display(df)
    print()

## 1.7 Literaturverzeichnis

Das Paper hat **106 Referenzen** im Format `[n]`. Wir extrahieren und zählen sie.

In [ ]:
# Literaturverzeichnis-Abschnitt finden
m = re.search(r'\nREFERENCES\s*\n(.*)', full_text, re.DOTALL)
if not m:
    m = re.search(r'\nReferences\s*\n(.*)', full_text, re.DOTALL)

refs = []
if m:
    refs = re.split(r'\n(?=\[\d+\])', m.group(1))
    refs = [r.strip() for r in refs if r.strip() and re.match(r'\[\d+\]', r)]

print(f'Referenzen gefunden: {len(refs)}  (erwartet: 106)')
print()
for r in refs[:4]:
    print(r[:130])
    print()

In [ ]:
def parse_year(s):
    m = re.search(r'\b(20\d{2}|19\d{2})\b', s)
    return int(m.group(1)) if m else None

year_counts = Counter(parse_year(r) for r in refs)
year_counts.pop(None, None)

print('Referenzen nach Erscheinungsjahr:')
print(pd.Series(year_counts).sort_index().to_string())

## 1.8 Bildextraktion

Das Paper enthält Abbildungen (Figures 1–5). `PyMuPDF` kann die eingebetteten Rasterbilder extrahieren.

In [ ]:
img_df = count_images(pdf_paths['arxiv'])
print('Bilder pro Seite:')
print(img_df[img_df['n_images']>0].to_string(index=False))
print(f"\nGesamt: {img_df['n_images'].sum()}")

## 1.9 Zusammenfassung — arXiv-Artikel

In [ ]:
print('=' * 55)
print('  EXTRAKTIONSZUSAMMENFASSUNG — arXiv-Artikel')
print('=' * 55)
display(extraction_report(pdf_paths['arxiv'])[['library','chars','words','time_sec']])
print(f'  Abstract gefunden   : {abstract_match is not None}')
print(f'  Abschnitte gefunden : {len(sec_df)}')
print(f'  Tabellen (plumber)  : {len(found_tables)}')
print(f'  Referenzen gefunden : {len(refs)}')
print(f'  Eingebettete Bilder : {img_df["n_images"].sum()}')
print()
print('Zentrale Beobachtungen:')
print('  • Alle vier Bibliotheken extrahieren Text gut (sauberes natives PDF)')
print('  • Zweispaltiges Layout kann linke/rechte Spalten im Output verschachteln')
print('  • pdfminer-Layoutobjekte legen die räumliche Struktur offen, um das zu korrigieren')
print('  • pdfplumber erkennt Tabellen über Liniengeometrie')
print('  • PyMuPDF ist am schnellsten und verarbeitet Bildextraktion sauber')

---
# Teil 2 — Swisscom Installationsanleitung (Selbst ausprobieren)

Dies ist eine **zweiseitige A4-Schnellstartanleitung** auf Deutsch für den Swisscom Internet-Box Glasfaser-Router.

Aus der visuellen Inspektion:
- **Seite 1:** Eine große zusammengesetzte Illustration mit allen 10 Installationsschritten. Schrittnummern und der Hauptteil des Anleitungstexts sind *im Bild* eingebettet.
- **Seite 2:** Dichterer Text — LED-Verhalten, WLAN-Einstellungen, rechtliche Hinweise.

**Aufgabe:** Die gleichen Analyse-Werkzeuge wie in Teil 1 anwenden und untersuchen, was bei der Extraktion erhalten bleibt.

## 2.0 Zuerst visuell inspizieren

In [ ]:
print('Seite 1:')
render_page(pdf_paths['swisscom'], page_number=0, dpi=90)
print('Seite 2:')
render_page(pdf_paths['swisscom'], page_number=1, dpi=90)

## ✍️ Aufgabe 2.1 — Bibliotheksvergleich

`extraction_report()` auf das Swisscom-PDF anwenden.
- Wie unterscheiden sich die Zeichenanzahlen im Vergleich zum arXiv-Ergebnis?
- Ist die Streuung zwischen den Bibliotheken größer oder kleiner?
- Das Dokument hat 2 Seiten, aber deutlich weniger Wörter als der arXiv-Artikel. Wirkt die Zeichenanzahl plausibel?

In [ ]:
# HIER CODE EINFÜGEN


## ✍️ Aufgabe 2.2 — Welcher Text überlebt auf Seite 1?

Die ersten 1500 Zeichen **nur von Seite 1** mit jeder Bibliothek extrahieren und ausgeben (Index `[0]` der Seitenliste verwenden).
- Mit dem gerenderten Bild vergleichen.
- Welcher Text wurde extrahiert? Wo befindet er sich physisch auf der Seite?
- Welche wichtigen Informationen fehlen vollständig?

In [ ]:
# Hinweis pdfplumber:  with pdfplumber.open(path) as pdf: text = pdf.pages[0].extract_text()
# Hinweis PyMuPDF:     doc = fitz.open(path); text = doc[0].get_text(); doc.close()

# HIER CODE EINFÜGEN



## ✍️ Aufgabe 2.3 — Bilder pro Seite

`count_images()` auf das Swisscom-PDF anwenden.
- Wie viele eingebettete Bilder gibt es pro Seite?
- Erklärt die Bildanzahl, warum die Textextraktion auf Seite 1 so wenig liefert?

In [ ]:
# HIER CODE EINFÜGEN



## ✍️ Aufgabe 2.4 — Layoutanalyse

`layout_elements()` auf Seite 1 des Swisscom-PDFs anwenden.
- Welche Elementtypen erscheinen? Den Mix aus `LTTextBox` / `LTFigure` / `LTRect` mit dem arXiv-Artikel vergleichen.
- Dieselbe Analyse auf Seite 2 anwenden — wie verändert sich die Verteilung der Elementtypen?
- Was deutet das Vorhandensein vieler `LTRect`-Elemente an? (Hinweis: Seite visuell betrachten.)

In [ ]:
# HIER CODE EINFÜGEN



## ✍️ Aufgabe 2.5 — Installationsschritte extrahieren (Textebene)

Trotz des bildlastigen Layouts ist der Schritttext *teilweise* als kurze deutsche Sätze in der PDF-Textebene vorhanden (z.B. *"Stecken Sie das Netzteil ein."*).

1. Zuerst den **rohen extrahierten Text** des gesamten Dokuments ausgeben — die tatsächlichen Muster untersuchen.
2. `extract_steps(pdf_path)` mit Regex schreiben, um schrittartige Sätze zu extrahieren.
3. Einen DataFrame mit den Spalten `schritt` (int) und `anweisung` (str) zurückgeben.

**Hinweis:** Die Schrittnummern (1–10) sind in den Bildern eingebettet, nicht in der Textebene — eine zuverlässige Zuordnung von Nummer zu Anweisung ist möglicherweise nicht möglich.

In [ ]:
# Schritt 1: Rohtext erkunden
raw = get_text(pdf_paths['swisscom'], library='pymupdf')
print(raw)

In [ ]:
def extract_steps(pdf_path):
    """
    Installationsschritte aus der Swisscom-Anleitung extrahieren.
    Gibt einen DataFrame zurück: schritt (int), anweisung (str)
    """
    # HIER CODE EINFÜGEN
    pass

result = extract_steps(pdf_paths['swisscom'])
if result is not None:
    display(result)

---
# Teil 3 — BSc BAI Modulübersicht (Selbst ausprobieren)

Dies ist ein **gestalterisch aufwändiges Layoutdokument** für den Studiengang BSc Business Artificial Intelligence an der FHNW.

Aus der visuellen Inspektion:
- **Seite 1:** Ein Modulraster — farbige Boxen (Farbe = Wissensbereich) in Semesterspalten angeordnet. Jede Box enthält einen Modulnamen und eine ECTS-Zahl.
- **Seite 2:** Blockdiagramme zur Vollzeit-/Teilzeit-Studienstruktur.

Die Textebene **enthält** die Modulnamen, aber die **räumlichen Beziehungen** (welche Semesterspalte, welcher Farb-/Wissensbereich) existieren nur im visuellen Layout — sie sind für die Standard-Textextraktion unsichtbar.

## 3.0 Zuerst visuell inspizieren

In [ ]:
print('Seite 1 — Modulraster:')
render_page(pdf_paths['curriculum'], page_number=0, dpi=110)
print('Seite 2 — Studienstruktur:')
render_page(pdf_paths['curriculum'], page_number=1, dpi=110)

## ✍️ Aufgabe 3.1 — Bibliotheksvergleich

`extraction_report()` auf das Curriculum-PDF anwenden.
- Die Zeichenausbeuten aller drei PDFs vergleichen.
- Liefert dieses Dokument mehr oder weniger Zeichen als angesichts des visuellen Inhalts erwartet?
- Gibt es eine Bibliothek, die deutlich von den anderen abweicht? Warum könnte das so sein?

In [ ]:
# HIER CODE EINFÜGEN



## ✍️ Aufgabe 3.2 — Welcher Text wurde extrahiert?

Den **vollständig extrahierten Text von Seite 1** mit PyMuPDF ausgeben und mit dem gerenderten Bild vergleichen.
- Welche Modulnamen erscheinen korrekt im Text?
- Ist die Reihenfolge sinnvoll? Spiegelt sie das visuelle Layout wider (links→rechts, oben→unten nach Semester)?
- Welche im Bild sichtbaren Informationen fehlen vollständig im Text?

In [ ]:
# HIER CODE EINFÜGEN



## ✍️ Aufgabe 3.3 — Modulnamen und ECTS-Punkte extrahieren

Obwohl das Layout verloren geht, enthält der Rohtext Modulnamen gefolgt von ECTS-Zahlen (3, 6, 9 oder 12).

`extract_modules(pdf_path)` schreiben, die:
1. Den vollständigen Text von Seite 1 extrahiert
2. Modulname + ECTS-Paare identifiziert
3. Einen DataFrame mit den Spalten `modulname` (str) und `ects` (int) zurückgibt

Das Ergebnis mit dem visuellen Inhalt abgleichen: Das Curriculum hat **~30 Module**. Wie viele findet der Extraktor? Was fehlt?

In [ ]:
# Zuerst den Rohtext erkunden
raw = get_text(pdf_paths['curriculum'], library='pymupdf')
print(raw[:3000])

In [ ]:
def extract_modules(pdf_path):
    """
    Modulnamen und ECTS aus dem Curriculum-PDF extrahieren.
    Gibt DataFrame zurück: modulname (str), ects (int)
    """
    # HIER CODE EINFÜGEN
    pass

mods = extract_modules(pdf_paths['curriculum'])
if mods is not None:
    print(f'Module gefunden: {len(mods)}')
    print(f'Gesamt-ECTS:     {mods["ects"].sum()}')
    display(mods)

## ✍️ Aufgabe 3.4 — Layoutanalyse: Spaltenstruktur ermitteln

`layout_elements()` auf Seite 1 des Curriculum-PDFs anwenden.
- x0- und y0-Koordinaten zusammen mit Textvorschauen aller `LTTextBox`-Elemente ausgeben.
- Die Seite ist A4-Querformat (~842 Punkte breit, ~595 Punkte hoch). Das Raster hat ~6 Semesterspalten.
- Lassen sich anhand der x0-Werte x-Bereichsgrenzen für jede Semesterspalte bestimmen?
- Ein Streudiagramm von x0 vs. y0 der Textboxen erstellen, um die Spaltenstruktur zu visualisieren.

In [ ]:
# HIER CODE EINFÜGEN



## ✍️ Aufgabe 3.5 (Herausforderung) — Das Modulraster rekonstruieren

Das Curriculum-Raster hat:
- **Spalten** = Semester (Herbst Sem. 1 → Frühling Sem. 6, von links nach rechts)
- **Zeilen** = Wissensbereiche (Künstliche Intelligenz, Wirtschaftsinformatik, Wirtschaft, Anwendung & Praxis, Interdisziplinär), getrennt durch vertikale Position

Mithilfe der `x0`/`y0`-Koordinaten aus `layout_elements()`:
1. x-Bereichsgrenzen für die Semesterspalten definieren
2. y-Bereichsgrenzen für die Wissensbereichszeilen definieren
3. Jede Textbox einer `(semester, wissensbereich)`-Zelle zuordnen
4. Das Ergebnis als Pivot-Tabelle / DataFrame zurückgeben

**Hinweis:** Die Seite rendern und Grenzen visuell abschätzen. Mit nur 2 Spalten beginnen, um die Schwellenwerte zu prüfen, bevor alle 6 bearbeitet werden.

In [ ]:
# HIER CODE EINFÜGEN



## 📝 Reflexion — Teil 3

In [ ]:
# 1. Die Farbe jeder Box kodiert den Wissensbereich (orange = KI, grün = Wirtschaft, usw.).
#    Kann eine der vier Bibliotheken die Füllfarbe eines PDF-Rechtecks/Boxelements extrahieren?
#    Falls nicht, was wäre nötig? (Hinweis: PyMuPDF 'drawings' / 'paths' nachschlagen)
#    Antwort: ...

# 2. Die räumliche Position einer Modulbox (welche Spalte, welche Zeile) kodiert Semester und
#    Wissensbereich — das ist bei der Standard-Textextraktion unsichtbar.
#    Welche Informationen lassen sich daher *ohne* Koordinatenanalyse nicht zuverlässig wiederherstellen?
#    Antwort: ...

# 3. Für ein Projekt, das Curriculum-Dokumente wie dieses automatisch verarbeiten muss:
#    Was wäre ein robusterer Ansatz als PDF-Text-/Koordinatenextraktion?
#    Antwort: ...

---
# Teil 4 — Alle drei PDFs im Vergleich

In [ ]:
all_reports = []
for label in ['arxiv','swisscom','curriculum']:
    r = extraction_report(pdf_paths[label])
    r.insert(0, 'pdf', label)
    all_reports.append(r)
combined = pd.concat(all_reports, ignore_index=True)

print('Extrahierte Zeichen pro Bibliothek und PDF:')
pivot = combined.pivot_table(index='library', columns='pdf', values='chars').rename_axis(None,axis=1)
display(pivot[['arxiv','swisscom','curriculum']])

print('\nBilder pro PDF:')
for label in ['arxiv','swisscom','curriculum']:
    total = count_images(pdf_paths[label])['n_images'].sum()
    pages = len(pypdf.PdfReader(pdf_paths[label]).pages)
    print(f'  {label:15s}: {total} images across {pages} pages')

## 🏁 Lernziele und Erkenntnisse

| PDF-Typ | Was Textextraktion liefert | Was fehlt | Empfohlenes Werkzeug |
|----------|---------------------------|-----------|----------------------|
| Wissenschaftlicher Artikel (arXiv) | Volltext, Abschnitte, Referenzen, Tabellen | Zweispaltige Lesereihenfolge (mit Koordinaten korrigierbar) | `pdfplumber` (Tabellen) + `PyMuPDF` (Geschwindigkeit) |
| Bedienungsanleitung (Swisscom) | Textebeneninhalt (Seite 2 gut, Seite 1 kaum) | Schritttext und -nummern in Bildern eingebettet | `PyMuPDF` + OCR für Bildinhalte |
| Gestalterisches Layout (BAI Curriculum) | Rohliste der Modulnamen + ECTS-Zahlen | Semester-/Bereichszuordnungen, Farben, räumliche Bedeutung | Koordinatenanalyse oder Render + Vision-Modell |

### Die grundlegende Erkenntnis

> Ein PDF ist ein Satz von **Darstellungsanweisungen**, kein Dokument.

Bedeutung, die durch Layout vermittelt wird — räumliche Position, Farbe, Gruppierung — ist **nicht in der Textebene** vorhanden. Je mehr ein Dokument auf Layout zur Bedeutungsvermittlung angewiesen ist, desto weniger kann die klassische Textextraktion erfassen.

```
Entscheidungsbaum für ein neues PDF:

  Ist die Textebene befüllt?
    NEIN → Gerenderte Seiten per OCR verarbeiten (pytesseract / easyocr / Cloud-Vision-API)
    JA   → Hängt die Bedeutung von räumlichem Layout oder Farbe ab?
              NEIN → Standard-Extraktion (PyMuPDF / pdfplumber) + Regex
              JA   → Seiten als Bilder rendern → Vision-Modell oder koordinatenbasiertes Parsing
```